# 04 — Governed Output Envelope

A **governed response** in Phionyx is not just text — it is a structured
artifact that carries the provenance of how it was produced: which gates
ran, what state the runtime was in, which kill-switch checks fired, and
a tamper-evident audit hash. Anything downstream (a frontend, a compliance
log, a human reviewer) can verify the response against this envelope.

This notebook builds one **without an LLM**. The narrative cell uses a
deterministic placeholder so the whole flow runs in seconds on a fresh
`pip install phionyx-core`. In production the placeholder is replaced by
a real model — the rest of the envelope shape is identical.

**Public API used:** `EchoState2`, `calculate_phi_v2_1`,
`calculate_phi_cognitive`, `KillSwitch`, `get_canonical_blocks`.


## 1. Imports

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import phionyx_core
from phionyx_core import (
    EchoState2,
    calculate_phi_v2_1,
    calculate_phi_cognitive,
)
from phionyx_core.governance.kill_switch import KillSwitch
from phionyx_core.contracts.telemetry import get_canonical_blocks

print("phionyx_core:", phionyx_core.__version__)

phionyx_core: 0.2.1


## 2. Input + safety gate

A safety gate is a deterministic function — no model, no fuzziness. It
rejects a known blocklist and flags inputs that fail length / charset
sanity. In the canonical pipeline this runs at position 3
(`input_safety_gate`).


In [2]:
USER_INPUT = "Walk me through the three layers Phionyx runs around an LLM."

BLOCKED_PATTERNS = ["ignore previous instructions", "system prompt"]

def input_safety_gate(text: str) -> dict:
    text_lower = text.lower()
    matched = [p for p in BLOCKED_PATTERNS if p in text_lower]
    if matched:
        return {"allowed": False, "reason": f"blocked patterns: {matched}"}
    if not text.strip() or len(text) > 4000:
        return {"allowed": False, "reason": "length out of range"}
    return {"allowed": True, "reason": None}

safety = input_safety_gate(USER_INPUT)
print(safety)

{'allowed': True, 'reason': None}


## 3. Build a state vector from input heuristics

In a real run the state vector is updated by upstream blocks (sentiment,
arousal estimation, prior-turn memory). Here we derive a plausible state
from string heuristics so the demo is self-contained and deterministic.


In [3]:
def state_from_input(text: str) -> EchoState2:
    # Cheap deterministic features.
    word_count = len(text.split())
    has_question = "?" in text
    has_negative = any(w in text.lower() for w in ["hate", "angry", "broken", "bad"])

    arousal = min(1.0, 0.4 + 0.05 * has_question + 0.02 * (word_count > 12))
    valence = -0.4 if has_negative else 0.2
    entropy = max(0.1, min(0.6, word_count / 50.0))

    return EchoState2(A=arousal, V=valence, H=entropy)


state = state_from_input(USER_INPUT)
print(f"state  A={state.A}  V={state.V}  H={state.H}")
print(f"derived  resonance={state.resonance:.3f}  stability={state.stability:.3f}")

state  A=0.4  V=0.2  H=0.22
derived  resonance=0.320  stability=0.780


## 4. Φ + entropy from physics

Standard `calculate_phi_v2_1`. The `phi_cognitive` floor (Base Life
Support) is what stops a high-stress / negative-valence input from
collapsing the runtime to zero.


In [4]:
phi = calculate_phi_v2_1(
    valence=state.V,
    arousal=state.A,
    amplitude=state.A * 10.0,
    time_delta=0.1,
    gamma=0.15,
    stability=state.stability,
    entropy=state.H,
    w_c=0.6,
    w_p=0.4,
)

print("Φ components:")
for k, v in phi.items():
    print(f"  {k:<18} {v:.4f}")

Φ components:
  phi                0.7515
  phi_cognitive      0.2017
  phi_physical       1.5762
  weight_cognitive   0.6000
  weight_physical    0.4000


## 5. Kill-switch evaluation

Even with a synthetic narrative cell below, the kill-switch evaluation is
real. Inputs are bounded scalars, outputs are deterministic.


In [5]:
ks = KillSwitch()
ks_result = ks.evaluate(
    ethics_max_risk=0.10,        # placeholder for the ethics gate output
    t_meta=0.85,                 # meta-cognitive trust
    drift_detected=False,
    turn_id=1,
)

print(f"kill switch state: {ks.state.value}")
print(f"triggered:         {ks_result.triggered}")
print(f"reason:            {ks_result.reason}")

kill switch state: armed
triggered:         False
reason:            All safety checks passed


## 6. Narrative layer — placeholder

In production this cell calls the LLM. For demo purposes we use a
deterministic echo so the notebook output is reproducible and so the
envelope shape can be verified without a model dependency.


In [6]:
def narrative_layer_synthetic(user_input: str, phi_total: float) -> str:
    return (
        f"(synthetic) You asked about Phionyx layers. Φ = {phi_total:.3f}; "
        "the runtime resolved this turn through the deterministic substrate, "
        "the governance gates, and the audit layer."
    )


response_text = narrative_layer_synthetic(USER_INPUT, phi["phi"])
print(response_text)

(synthetic) You asked about Phionyx layers. Φ = 0.751; the runtime resolved this turn through the deterministic substrate, the governance gates, and the audit layer.


## 7. Build the governed envelope

The envelope is a flat, JSON-serialisable dict that captures every piece
the demo just produced. In production this would be a Pydantic schema
with validation; here a plain dict keeps the shape transparent.


In [7]:
def audit_hash(payload: dict) -> str:
    # Canonical JSON, then SHA-256. In production: Ed25519-signed
    # AuditRecord with hash chain to prior turn.
    canonical = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest()


timestamp = datetime(2026, 5, 1, 0, 0, 0, tzinfo=timezone.utc).isoformat()

envelope = {
    "schema_version": "phionyx-governed-response/0.1",
    "phionyx_core_version": phionyx_core.__version__,
    "turn_id": 1,
    "timestamp_utc": timestamp,
    "input": {
        "user_text": USER_INPUT,
        "safety": safety,
    },
    "state": {
        "arousal": state.A,
        "valence": state.V,
        "entropy": state.H,
        "resonance": round(state.resonance, 6),
        "stability": round(state.stability, 6),
    },
    "phi": {k: round(v, 6) for k, v in phi.items()},
    "governance": {
        "kill_switch_state": ks.state.value,
        "kill_switch_triggered": ks_result.triggered,
        "kill_switch_reason": ks_result.reason,
        "ethics_max_risk": 0.10,
        "t_meta": 0.85,
        "drift_detected": False,
    },
    "response": {
        "text": response_text,
        "narrative_layer": "synthetic_echo",
    },
    "pipeline": {
        "contract_version": "3.8.0",
        "block_count": len(get_canonical_blocks()),
    },
}

# Audit hash is the SHA-256 of everything above. Include it last so it
# covers the rest of the envelope deterministically.
envelope["audit"] = {
    "hash_alg": "sha256",
    "envelope_hash": audit_hash(envelope),
}

print(json.dumps(envelope, indent=2))

{
  "schema_version": "phionyx-governed-response/0.1",
  "phionyx_core_version": "0.2.1",
  "turn_id": 1,
  "timestamp_utc": "2026-05-01T00:00:00+00:00",
  "input": {
    "user_text": "Walk me through the three layers Phionyx runs around an LLM.",
    "safety": {
      "allowed": true,
      "reason": null
    }
  },
  "state": {
    "arousal": 0.4,
    "valence": 0.2,
    "entropy": 0.22,
    "resonance": 0.32,
    "stability": 0.78
  },
  "phi": {
    "phi": 0.751476,
    "phi_cognitive": 0.201674,
    "phi_physical": 1.576179,
    "weight_cognitive": 0.6,
    "weight_physical": 0.4
  },
  "governance": {
    "kill_switch_state": "armed",
    "kill_switch_triggered": false,
    "kill_switch_reason": "All safety checks passed",
    "ethics_max_risk": 0.1,
    "t_meta": 0.85,
    "drift_detected": false
  },
  "response": {
    "text": "(synthetic) You asked about Phionyx layers. \u03a6 = 0.751; the runtime resolved this turn through the deterministic substrate, the governance gates, a

## 8. Persist + verify

The envelope is saved to `examples/envelopes/governed_response.json`.
Re-loading and re-hashing must produce the same digest — that's the
tamper-evident property in miniature.


In [8]:
out_path = Path("../envelopes/governed_response.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(envelope, indent=2) + "\n", encoding="utf-8")
print(f"wrote {out_path.resolve()}")

# Verify round trip.
loaded = json.loads(out_path.read_text(encoding="utf-8"))
saved_hash = loaded.pop("audit")["envelope_hash"]
recomputed = audit_hash(loaded)
print(f"saved hash      : {saved_hash}")
print(f"recomputed hash : {recomputed}")
assert saved_hash == recomputed, "audit hash drift — envelope was tampered with"
print("\nverified: envelope is intact")

wrote /tmp/phionyx-research/examples/envelopes/governed_response.json
saved hash      : b3ea0af3e3efba99a4cb8da957afd3007ad6e65b7919b7981ba0d7e2f59ce41d
recomputed hash : b3ea0af3e3efba99a4cb8da957afd3007ad6e65b7919b7981ba0d7e2f59ce41d

verified: envelope is intact


## What this proves

- The envelope shape is reproducible from the public API.
- Every governance decision (safety gate, kill switch, Φ) is captured
  alongside the response, not behind it.
- A tamper-evident hash covers the whole envelope deterministically.
- None of this requires an LLM call. The narrative cell is a synthetic
  placeholder; replacing it with a real model leaves the rest of the
  envelope shape unchanged.

In production:

- Replace the synthetic narrative with an LLM provider.
- Replace the SHA-256 hash with `phionyx_core.contracts.v4.audit_record.AuditRecord`,
  which adds Ed25519 signing and a chain reference to the previous turn.
- Replace the dict with a Pydantic schema (versioned via
  `schema_version`).

The committed sample at
[`examples/envelopes/governed_response.json`](../envelopes/governed_response.json)
is the exact output of this notebook on `phionyx-core 0.2.1`.
